In [1]:

import pandas as pd
import numpy as np
import pickle
from collections import defaultdict
import matplotlib.pyplot as plt

# Load the N=10^7 peak data
n107_results = pd.read_csv('N10^7_detailed_results.csv')
print("N=10^7 Peak Data:")
print(n107_results)
print("\nLiouville peaks at N=10^7:")
liouville_107 = n107_results[n107_results['Function'] == 'Liouville']
print(liouville_107)


N=10^7 Peak Data:
 Function N t_value peak_height r_value
0 Zeta 10000000 1.848485e+07 16.508561 0.780727
1 Zeta 10000000 1.232323e+07 10.284027 0.631713
2 Zeta 10000000 1.090909e+07 6.367052 -0.074793
3 Zeta 10000000 1.616162e+07 5.935386 1.437305
4 Zeta 10000000 1.191919e+07 5.878669 0.669312
5 Liouville 10000000 1.979798e+07 21.827989 1.924900
6 Liouville 10000000 1.787879e+07 17.305548 1.181383
7 Liouville 10000000 1.363636e+07 12.219003 1.787888
8 Liouville 10000000 1.929293e+07 9.473383 -0.345984
9 Liouville 10000000 1.212121e+07 7.941801 0.850410

Liouville peaks at N=10^7:
 Function N t_value peak_height r_value
5 Liouville 10000000 1.979798e+07 21.827989 1.924900
6 Liouville 10000000 1.787879e+07 17.305548 1.181383
7 Liouville 10000000 1.363636e+07 12.219003 1.787888
8 Liouville 10000000 1.929293e+07 9.473383 -0.345984
9 Liouville 10000000 1.212121e+07 7.941801 0.850410


In [2]:

# Load omega values for N=10^6 and N=10^7
print("Loading omega values...")
with open('omega_values_N1e6.pkl', 'rb') as f:
 omega_N1e6 = pickle.load(f)
print(f"omega_N1e6 shape: {omega_N1e6.shape}")

with open('omega_values_N1e7.pkl', 'rb') as f:
 omega_N1e7 = pickle.load(f)
print(f"omega_N1e7 shape: {omega_N1e7.shape}")

# Remember: 0-based indexing, omega_values[i] = Ω(i+1)
print(f"\nSample omega values (1-indexed n, Ω(n)):")
for n in [1, 2, 3, 4, 5, 6, 12, 24]:
 print(f" Ω({n}) = {omega_N1e6[n-1]}")


Loading omega values...
omega_N1e6 shape: (1000000,)
omega_N1e7 shape: (10000000,)

Sample omega values (1-indexed n, Ω(n)):
 Ω(1) = 0
 Ω(2) = 1
 Ω(3) = 1
 Ω(4) = 2
 Ω(5) = 1
 Ω(6) = 2
 Ω(12) = 3
 Ω(24) = 4


In [3]:

# Define the Liouville function
def liouville_coefficient(n, omega_n):
 """
 Liouville function: λ(n) = (-1)^Ω(n)
 omega_n should be the value Ω(n) for the input n
 """
 return (-1.0)**omega_n

def compute_dirichlet_polynomial_with_decomposition(t, N, omega_values, coeff_func):
 """
 Compute D_F(t; N) with full ω-class decomposition using Kahan summation.
 
 Parameters:
 - t: evaluation point
 - N: truncation level
 - omega_values: array where omega_values[i] = Ω(i+1)
 - coeff_func: function(n, omega_n) -> coefficient a_n
 
 Returns:
 - D: total Dirichlet polynomial value
 - S_k_dict: dictionary mapping k -> S_k (complex)
 - omega_class_counts: dictionary mapping k -> count of terms in that class
 """
 # Initialize dictionaries for ω-class sums using Kahan summation
 S_k = defaultdict(complex)
 S_k_c = defaultdict(float) # Kahan compensation for real part
 S_k_ci = defaultdict(float) # Kahan compensation for imaginary part
 omega_counts = defaultdict(int)
 
 for n in range(1, N+1):
 omega_n = omega_values[n-1] # 0-based indexing
 a_n = coeff_func(n, omega_n)
 
 # Compute n^{-1/2 - it} = n^{-1/2} * e^{-it*log(n)}
 # = n^{-1/2} * (cos(-t*log(n)) + i*sin(-t*log(n)))
 n_sqrt = np.sqrt(n)
 log_n = np.log(n)
 angle = -t * log_n
 
 term_real = a_n / n_sqrt * np.cos(angle)
 term_imag = a_n / n_sqrt * np.sin(angle)
 
 # Kahan summation for real part
 y_r = term_real - S_k_c[omega_n]
 t_r = S_k[omega_n].real + y_r
 S_k_c[omega_n] = (t_r - S_k[omega_n].real) - y_r
 
 # Kahan summation for imaginary part
 y_i = term_imag - S_k_ci[omega_n]
 t_i = S_k[omega_n].imag + y_i
 S_k_ci[omega_n] = (t_i - S_k[omega_n].imag) - y_i
 
 S_k[omega_n] = complex(t_r, t_i)
 omega_counts[omega_n] += 1
 
 # Total Dirichlet polynomial
 D = sum(S_k.values())
 
 return D, dict(S_k), omega_counts

print("Dirichlet polynomial computation function with Kahan summation defined.")


Dirichlet polynomial computation function with Kahan summation defined.


In [4]:

def compute_canonical_r_with_decomposition(S_k_dict):
 """
 Compute the canonical r metric with full pairwise decomposition.
 
 r = Num / Denom
 Num = Σ_{j≠k} Re[S_j S̄_k]
 Denom = Σ_k |S_k|²
 
 Also decompose Num into:
 - Adjacent pairs (Δω = 1): j and k differ by 1
 - Non-adjacent pairs (Δω > 1): j and k differ by more than 1
 
 Returns:
 - r: canonical r metric
 - Num: numerator
 - Denom: denominator
 - pairwise_contributions: dict mapping (j,k) -> Re[S_j S̄_k]
 - adjacent_sum: sum of contributions from adjacent pairs
 - nonadjacent_sum: sum of contributions from non-adjacent pairs
 """
 # Compute denominator
 Denom = sum(abs(S_k)**2 for S_k in S_k_dict.values())
 
 # Compute numerator and pairwise contributions
 pairwise = {}
 adjacent_sum = 0.0
 nonadjacent_sum = 0.0
 
 ks = sorted(S_k_dict.keys())
 
 for i, j in enumerate(ks):
 for k in ks[i+1:]: # Only count each pair once (j < k)
 # Contribution is Re[S_j S̄_k] + Re[S_k S̄_j] = 2*Re[S_j S̄_k]
 # So we store the contribution for the pair (j,k) as 2*Re[S_j S̄_k]
 contrib = 2 * (S_k_dict[j] * np.conj(S_k_dict[k])).real
 pairwise[(j, k)] = contrib
 
 delta_omega = abs(k - j)
 if delta_omega == 1:
 adjacent_sum += contrib
 else:
 nonadjacent_sum += contrib
 
 Num = adjacent_sum + nonadjacent_sum
 
 if Denom > 0:
 r = Num / Denom
 else:
 r = np.nan
 
 return r, Num, Denom, pairwise, adjacent_sum, nonadjacent_sum

print("Canonical r metric computation with pairwise decomposition defined.")


Canonical r metric computation with pairwise decomposition defined.


In [5]:

# Extract t values for Liouville at N=10^7
t_values_107 = liouville_107['t_value'].values
print("Liouville peak t-values at N=10^7:")
print(t_values_107)

# Compute ω-class decomposition for all 5 peaks at N=10^7
N_107 = 10**7
results_107 = []

print("\nComputing ω-class decompositions for N=10^7 peaks...")
for idx, t in enumerate(t_values_107):
 print(f"\nPeak {idx+1}: t = {t:.2f}")
 D, S_k, omega_counts = compute_dirichlet_polynomial_with_decomposition(
 t, N_107, omega_N1e7, liouville_coefficient
 )
 
 r, Num, Denom, pairwise, adj_sum, nonadj_sum = compute_canonical_r_with_decomposition(S_k)
 
 print(f" |D| = {abs(D):.6f}")
 print(f" r (canonical) = {r:.6f}")
 print(f" Num = {Num:.6f}, Denom = {Denom:.6f}")
 print(f" Adjacent sum = {adj_sum:.6f}, Non-adjacent sum = {nonadj_sum:.6f}")
 print(f" Number of ω-classes: {len(S_k)}")
 
 results_107.append({
 't': t,
 'D': D,
 'r': r,
 'Num': Num,
 'Denom': Denom,
 'S_k': S_k,
 'pairwise': pairwise,
 'adjacent_sum': adj_sum,
 'nonadjacent_sum': nonadj_sum
 })

print("\nAll N=10^7 peaks processed.")


Liouville peak t-values at N=10^7:
[19797979.7979798 17878787.87878788 13636363.63636364 19292929.29292929
 12121212.12121212]

Computing ω-class decompositions for N=10^7 peaks...

Peak 1: t = 19797979.80


 |D| = 21.827989
 r (canonical) = 4.243715
 Num = 385.597818, Denom = 90.863276
 Adjacent sum = 164.582687, Non-adjacent sum = 221.015131
 Number of ω-classes: 24

Peak 2: t = 17878787.88


 |D| = 17.305548
 r (canonical) = 5.914642
 Num = 256.170726, Denom = 43.311282
 Adjacent sum = 80.595498, Non-adjacent sum = 175.575228
 Number of ω-classes: 24

Peak 3: t = 13636363.64


 |D| = 12.219003
 r (canonical) = 2.715534
 Num = 109.120300, Denom = 40.183743
 Adjacent sum = 67.290466, Non-adjacent sum = 41.829834
 Number of ω-classes: 24

Peak 4: t = 19292929.29


 |D| = 9.473383
 r (canonical) = 1.785574
 Num = 57.527221, Denom = 32.217768
 Adjacent sum = 39.228831, Non-adjacent sum = 18.298390
 Number of ω-classes: 24

Peak 5: t = 12121212.12


 |D| = 7.941801
 r (canonical) = 3.748738
 Num = 49.790319, Denom = 13.281889
 Adjacent sum = 22.897272, Non-adjacent sum = 26.893047
 Number of ω-classes: 24

All N=10^7 peaks processed.


In [6]:

# Compare to the r values in the CSV (which are non-canonical)
print("Comparison of canonical r vs. CSV r values at N=10^7:")
print("Peak | t_value | r (canonical) | r (CSV) | Ratio")
print("-" * 65)
for idx, (result, csv_r) in enumerate(zip(results_107, liouville_107['r_value'].values)):
 ratio = result['r'] / csv_r if csv_r != 0 else np.nan
 print(f"{idx+1:4d} | {result['t']:13.2f} | {result['r']:13.6f} | {csv_r:9.6f} | {ratio:6.3f}")

# Note: The CSV r values are computed with a non-canonical method
# Our canonical r values are much larger, confirming the warning


Comparison of canonical r vs. CSV r values at N=10^7:
Peak | t_value | r (canonical) | r (CSV) | Ratio
-----------------------------------------------------------------
 1 | 19797979.80 | 4.243715 | 1.924900 | 2.205
 2 | 17878787.88 | 5.914642 | 1.181383 | 5.007
 3 | 13636363.64 | 2.715534 | 1.787888 | 1.519
 4 | 19292929.29 | 1.785574 | -0.345984 | -5.161
 5 | 12121212.12 | 3.748738 | 0.850410 | 4.408


In [7]:

# Let me try an even more optimized approach - compute in batches and use numba if needed
# First, let's try with just 50 grid points to see if we can get any results

N_106 = 10**6
t_min = N_106
t_max = 2 * N_106
n_grid = 50 # Minimal grid

print(f"Testing with minimal grid: {n_grid} points")
print(f"Range: t ∈ [{t_min}, {t_max}]")

t_grid = np.linspace(t_min, t_max, n_grid)

# Precompute coefficients and normalization for Liouville at N=10^6
omega_n = omega_N1e6[:N_106]
a_n = (-1.0)**omega_n
n_array = np.arange(1, N_106+1)
coeffs = a_n / np.sqrt(n_array)
log_n = np.log(n_array)

print("Precomputation done. Starting grid evaluation...")

import time
start = time.time()

magnitudes = []
for i, t in enumerate(t_grid):
 angle = -t * log_n
 D_real = np.sum(coeffs * np.cos(angle))
 D_imag = np.sum(coeffs * np.sin(angle))
 mag = np.sqrt(D_real**2 + D_imag**2)
 magnitudes.append(mag)
 
 if (i+1) % 10 == 0:
 elapsed = time.time() - start
 rate = (i+1) / elapsed
 remaining = (n_grid - i - 1) / rate
 print(f" {i+1}/{n_grid} done, {elapsed:.1f}s elapsed, ~{remaining:.1f}s remaining")

magnitudes = np.array(magnitudes)
elapsed = time.time() - start
print(f"\nGrid search completed in {elapsed:.1f} seconds")

# Find top peaks
n_peaks = 10
peak_indices = np.argsort(magnitudes)[::-1][:n_peaks]
peak_t_candidates = t_grid[peak_indices]
peak_mags_candidates = magnitudes[peak_indices]

print(f"\nTop {n_peaks} peak candidates:")
for i in range(n_peaks):
 print(f"{i+1:2d}. t = {peak_t_candidates[i]:12.2f}, |D| = {peak_mags_candidates[i]:8.4f}")


TimeoutError: Code execution timed out after 751 seconds